# درس 7: کوواریانس و همبستگی

متن کامل درس: [https://quantacademy.ir/courses/quant-math/covariance-correlation/](https://quantacademy.ir/courses/quant-math/covariance-correlation/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. کوواریانس و همبستگی

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

URL = "https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv"
close = pd.read_csv(URL, parse_dates=["date"], index_col="date")["close"]
r = np.log(close).diff().dropna()
n = len(r)

rng = np.random.default_rng(21)
rho_true = 0.6
x = r.values
noise = rng.standard_normal(n) * x.std()
y = rho_true * x + np.sqrt(1 - rho_true**2) * noise

print(np.cov(x, y).round(6))
print(np.corrcoef(x, y).round(3))
print(round(np.cov(x, y)[0, 1] / (x.std(ddof=1) * y.std(ddof=1)), 3))

## ۲. همبستگی را با چه خطایی تخمین می‌زنیم؟

In [ ]:
rng = np.random.default_rng(22)
for size in [30, 90, 365, 1825]:
    a = rng.standard_normal((4000, size))
    b = rho_true * a + np.sqrt(1 - rho_true**2) * rng.standard_normal((4000, size))
    a = a - a.mean(axis=1, keepdims=True)
    b = b - b.mean(axis=1, keepdims=True)
    est = (a * b).sum(axis=1) / np.sqrt((a * a).sum(axis=1) * (b * b).sum(axis=1))
    lo, hi = np.percentile(est, [2.5, 97.5])
    print(size, round(est.mean(), 3), round(est.std(), 3), round(lo, 2), round(hi, 2), round((1 - rho_true**2) / np.sqrt(size), 3))

In [ ]:
def fisher_ci(rho, n, z=1.96):
    center = np.arctanh(rho)
    half = z / np.sqrt(n - 3)
    return np.tanh(center - half), np.tanh(center + half)

for size in [30, 365]:
    lo, hi = fisher_ci(0.6, size)
    print(size, round(lo, 2), round(hi, 2))

## ۳. همبستگی «عوض نشد»، فقط نویز بود

In [ ]:
rng = np.random.default_rng(23)
a = rng.standard_normal(n)
b = rho_true * a + np.sqrt(1 - rho_true**2) * rng.standard_normal(n)
pair = pd.DataFrame({"a": a, "b": b})
rolling = pair["a"].rolling(90).corr(pair["b"]).dropna()
print(round(pair["a"].corr(pair["b"]), 3), round(rolling.min(), 2), round(rolling.max(), 2))

## ۴. پیرسون در برابر اسپیرمن

In [ ]:
rng = np.random.default_rng(24)
a = rng.standard_normal(500)
b = rng.standard_normal(500)
print(round(stats.pearsonr(a, b)[0], 3), round(stats.spearmanr(a, b)[0], 3))
a[0], b[0] = 40.0, 40.0
print(round(stats.pearsonr(a, b)[0], 3), round(stats.spearmanr(a, b)[0], 3))

## ۵. همبستگی را روی قیمت‌ها حساب نکن

In [ ]:
rng = np.random.default_rng(25)
levels = rng.standard_normal((500, 2, n)).cumsum(axis=2)
rets = np.diff(levels, axis=2)

def corr_pairs(arr):
    a = arr[:, 0, :] - arr[:, 0, :].mean(axis=1, keepdims=True)
    b = arr[:, 1, :] - arr[:, 1, :].mean(axis=1, keepdims=True)
    return (a * b).sum(axis=1) / np.sqrt((a * a).sum(axis=1) * (b * b).sum(axis=1))

c_levels, c_rets = corr_pairs(levels), corr_pairs(rets)
print(round((np.abs(c_levels) > 0.5).mean(), 3), round(np.abs(c_levels).mean(), 3))
print(round((np.abs(c_rets) > 0.1).mean(), 3), round(np.abs(c_rets).mean(), 3))

## ۶. همبستگی صفر یعنی استقلال؟

In [ ]:
rng = np.random.default_rng(26)
u = rng.standard_normal(100_000)
v = u**2
print(round(stats.pearsonr(u, v)[0], 3), round(stats.spearmanr(u, v)[0], 3))

absr = r.abs()
print(round(r.corr(r.shift(1)), 3), round(absr.corr(absr.shift(1)), 3))